# Python `getattr()` 详解

`getattr(object, name, default)` 是 Python 内置函数，用来**安全地获取对象的属性**。

在 BioJSON 项目的 `token_tracker.py` 中用到了它：
```python
usage = getattr(response, "usage", None)
```

## 1. 基本语法

```python
getattr(object, name)          # 获取属性，不存在则报 AttributeError
getattr(object, name, default) # 获取属性，不存在则返回 default（不报错）
```

| 参数 | 含义 |
|------|------|
| `object` | 要从中获取属性的对象 |
| `name` | 属性名（字符串） |
| `default` | 可选，属性不存在时返回的默认值 |

## 2. 直接访问 vs getattr

In [1]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

buddy = Dog("Buddy", 3)

# === 直接访问 ===
print(f"buddy.name = {buddy.name}")   # Buddy
print(f"buddy.age = {buddy.age}")     # 3

# buddy.weight  # 报错！AttributeError: 'Dog' object has no attribute 'weight'

buddy.name = Buddy
buddy.age = 3


In [2]:
# === 用 getattr 访问 ===

# 属性存在：和直接访问效果一样
print(getattr(buddy, 'name'))         # Buddy
print(getattr(buddy, 'age'))          # 3

# 属性不存在 + 提供默认值：返回默认值，不报错
print(getattr(buddy, 'weight', 0))    # 0
print(getattr(buddy, 'color', None))  # None

# 属性不存在 + 不提供默认值：仍然报错
try:
    getattr(buddy, 'weight')
except AttributeError as e:
    print(f"报错了: {e}")

Buddy
3
0
None
报错了: 'Dog' object has no attribute 'weight'


## 3. `getattr` 和 `.` 的等价关系

```python
# 这两行完全等价：
buddy.name
getattr(buddy, 'name')
```

那 getattr 有什么额外好处呢？

| 场景 | `.` 访问 | `getattr` |
|------|---------|-----------|
| 属性名是变量 | 做不到 | ✅ 可以 |
| 属性可能不存在 | 必须 try/except | ✅ 提供 default 即可 |
| 属性名在运行时才知道 | 做不到 | ✅ 可以 |

In [ ]:
# 场景1：属性名是变量

attr_name = 'name'  # 属性名存在变量里

# buddy.attr_name   # 错！Python 会去找 attr_name 属性
print(getattr(buddy, attr_name))  # 正确！用变量值 'name' 去找

# 实际用途：动态访问不同属性
for attr in ['name', 'age']:
    print(f"  {attr} = {getattr(buddy, attr)}")

In [3]:
print(getattr(buddy, 'weight', '未知'))  # weight 不存在，返回 '未知'

未知


In [ ]:
# 场景2：属性可能不存在（防御性编程）

# 不用 getattr：需要 try/except
try:
    w = buddy.weight
except AttributeError:
    w = 0
print(f"方式1: weight = {w}")

# 用 getattr：一行搞定
w = getattr(buddy, 'weight', 0)
print(f"方式2: weight = {w}")

## 4. 在 TokenTracker 中的实际用法

```python
def add(self, response, stage, file, gene):
    usage = getattr(response, "usage", None)
    if usage is None:
        return   # 安全退出，不崩溃
    # ... 继续处理 usage.prompt_tokens 等
```

**为什么不直接写 `response.usage`？**

因为 API 调用可能失败或返回异常对象，这时 response 可能没有 `usage` 属性。
用 `getattr` + `None` 默认值，可以优雅地处理这种情况，而不是让整个程序崩溃。

In [4]:
# 模拟 TokenTracker 的场景

class NormalResponse:
    def __init__(self):
        self.usage = {"prompt_tokens": 100, "completion_tokens": 50}

class BrokenResponse:
    pass  # 没有 usage 属性

for resp in [NormalResponse(), BrokenResponse()]:
    usage = getattr(resp, "usage", None)
    if usage is None:
        print(f"  {type(resp).__name__}: 没有 usage，跳过")
    else:
        print(f"  {type(resp).__name__}: usage = {usage}")

  NormalResponse: usage = {'prompt_tokens': 100, 'completion_tokens': 50}
  BrokenResponse: 没有 usage，跳过


## 5. 小结

| 要点 | 说明 |
|------|------|
| 基本功能 | 安全地获取对象属性 |
| 三个参数 | `getattr(对象, '属性名', 默认值)` |
| 核心优势 | 属性不存在时返回默认值而不是报错 |
| 额外优势 | 属性名可以是变量（动态访问） |
| 项目中用途 | `token_tracker.py` 中安全获取 `response.usage` |